# 06 — Game Predictions

This notebook converts the 2026 team strength ratings into game level predictions.

Each matchup begins with the difference between the two teams' neutral field strength ratings. Home field advantage and schedule context are then incorporated to estimate an expected point differential and win probability for every game on the 2026 NFL schedule.

These game probabilities will become the inputs to the season simulation in Notebook 07.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import nflreadpy as nfl

In [2]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

In [3]:
team_strength = pd.read_parquet(
    PROCESSED_DIR / "2026_team_strength.parquet"
)

schedule = pl.read_parquet(
    PROCESSED_DIR / "schedule_clean.parquet"
)

In [4]:
print(schedule.columns)
print()
print(
    schedule
    .filter(pl.col("season") == 2026)
    .head(5)
)

['game_id', 'season', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'away_rest', 'home_rest']

shape: (0, 16)
┌─────────┬────────┬──────┬─────────┬───┬───────┬──────────┬───────────┬───────────┐
│ game_id ┆ season ┆ week ┆ gameday ┆ … ┆ total ┆ overtime ┆ away_rest ┆ home_rest │
│ ---     ┆ ---    ┆ ---  ┆ ---     ┆   ┆ ---   ┆ ---      ┆ ---       ┆ ---       │
│ str     ┆ i32    ┆ i32  ┆ str     ┆   ┆ i32   ┆ i32      ┆ i32       ┆ i32       │
╞═════════╪════════╪══════╪═════════╪═══╪═══════╪══════════╪═══════════╪═══════════╡
└─────────┴────────┴──────┴─────────┴───┴───────┴──────────┴───────────┴───────────┘


## 2026 Schedule

The historical schedule dataset currently ends with the 2025 season, so the 2026 schedule is loaded separately for game prediction.

The schedule provides each matchup's week, home and away teams, location, and rest information. These factors allow the neutral field team ratings to be converted into game specific predictions.

In [5]:
schedule_2026 = nfl.load_schedules(
    seasons=[2026]
)

schedule_2026 = schedule_2026.filter(
    pl.col("game_type") == "REG"
)

### Game Schedule

Only information known independently of game outcomes is retained for prediction.

Each game includes the matchup, location, and rest available to each team. Betting market information is intentionally excluded from the prediction features so that the model produces independent estimates.

In [6]:
games_2026 = (
    schedule_2026
    .select([
        "game_id",
        "season",
        "week",
        "gameday",
        "weekday",
        "gametime",
        "away_team",
        "home_team",
        "location",
        "away_rest",
        "home_rest"
    ])
    .sort([
        "week",
        "gameday",
        "gametime"
    ])
    .to_pandas()
)

## Matchup Strength

Each team's neutral field strength rating is attached to the schedule.

The difference between the home and away ratings provides the starting point for each game prediction before home field advantage and rest are considered.

In [7]:
strength_lookup = team_strength[
    [
        "team",
        "team_strength"
    ]
]

games_2026 = (
    games_2026
    .merge(
        strength_lookup.rename(columns={
            "team": "home_team",
            "team_strength": "home_team_strength"
        }),
        on="home_team",
        how="left"
    )
    .merge(
        strength_lookup.rename(columns={
            "team": "away_team",
            "team_strength": "away_team_strength"
        }),
        on="away_team",
        how="left"
    )
)

games_2026["neutral_strength_diff"] = (
    games_2026["home_team_strength"]
    - games_2026["away_team_strength"]
)

## Home-Field Advantage

Home field advantage is estimated directly from historical regular season results rather than assigning a fixed value manually.

Neutral site games are excluded from this estimate because neither team has a conventional home field advantage.

In [8]:
historical_schedule = (
    schedule
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & (pl.col("location") == "Home")
        & pl.col("result").is_not_null()
    )
)

HOME_FIELD_ADVANTAGE = (
    historical_schedule
    .select(
        pl.col("result").mean()
    )
    .item()
)

print(
    f"Historical home-field advantage: "
    f"{HOME_FIELD_ADVANTAGE:.2f} points"
)

Historical home-field advantage: 1.76 points


In [9]:
games_2026["home_field_adjustment"] = np.where(
    games_2026["location"] == "Home",
    HOME_FIELD_ADVANTAGE,
    0.0
)

In [10]:
# TEMPORARY / DELETE AFTER CHECK — Inspect 2026 rest differences

games_2026["rest_diff"] = (
    games_2026["home_rest"]
    - games_2026["away_rest"]
)

print(
    games_2026["rest_diff"]
    .value_counts()
    .sort_index()
)

print()
print(
    "Missing home ratings:",
    games_2026["home_team_strength"].isna().sum()
)

print(
    "Missing away ratings:",
    games_2026["away_team_strength"].isna().sum()
)

print(
    "Home-field advantage:",
    round(HOME_FIELD_ADVANTAGE, 3)
)

rest_diff
-7     12
-6      1
-4      3
-3     17
-2      4
-1     20
 0    162
 1     15
 2      2
 3     18
 4      2
 7     16
Name: count, dtype: int64

Missing home ratings: 0
Missing away ratings: 0
Home-field advantage: 1.764


## Rest Advantage

Rest differences can affect game performance, particularly around short weeks and bye weeks.

The effect is estimated from historical regular season games. Rest difference is capped at seven days so that unusual scheduling values do not have excessive influence on the prediction.

In [11]:
historical_rest = (
    schedule
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
        & pl.col("home_rest").is_not_null()
        & pl.col("away_rest").is_not_null()
    )
    .select([
        "result",
        "home_rest",
        "away_rest",
        "location"
    ])
    .to_pandas()
)

historical_rest["rest_diff"] = (
    historical_rest["home_rest"]
    - historical_rest["away_rest"]
).clip(-7, 7)

historical_rest["home_field"] = (
    historical_rest["location"] == "Home"
).astype(int)

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression

In [13]:
rest_model = LinearRegression()

rest_model.fit(
    historical_rest[
        [
            "home_field",
            "rest_diff"
        ]
    ],
    historical_rest["result"]
)

REST_POINT_VALUE = rest_model.coef_[1]

print(
    f"Estimated rest effect: "
    f"{REST_POINT_VALUE:.3f} points per extra rest day"
)

Estimated rest effect: 0.181 points per extra rest day


In [14]:
games_2026["rest_diff"] = (
    games_2026["home_rest"]
    - games_2026["away_rest"]
).clip(-7, 7)

games_2026["rest_adjustment"] = (
    games_2026["rest_diff"]
    * REST_POINT_VALUE
)

## Expected Game Margin

The expected margin combines three components:

- the difference in neutral-field team strength,
- home-field advantage,
- and the difference in rest between the two teams.

A positive expected margin favors the home team, while a negative expected margin favors the away team.

In [15]:
games_2026["expected_home_margin"] = (
    games_2026["neutral_strength_diff"]
    + games_2026["home_field_adjustment"]
    + games_2026["rest_adjustment"]
)

## Win Probability

Team strength provides an expected scoring margin, but individual NFL games contain substantial uncertainty.

Historical game margin variability is used to translate the expected margin into a win probability. This prevents small projected advantages from producing unrealistically confident predictions.

In [16]:
from scipy.stats import norm

In [17]:
historical_margin_std = (
    schedule
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
    )
    .select(
        pl.col("result").std()
    )
    .item()
)

print(
    f"Historical game-margin standard deviation: "
    f"{historical_margin_std:.2f} points"
)

Historical game-margin standard deviation: 14.20 points


In [18]:
games_2026["home_win_probability"] = norm.cdf(
    games_2026["expected_home_margin"]
    / historical_margin_std
)

games_2026["away_win_probability"] = (
    1 - games_2026["home_win_probability"]
)

In [19]:
week_1_predictions = games_2026[
    [
        "away_team",
        "home_team",
        "neutral_strength_diff",
        "home_field_adjustment",
        "rest_adjustment",
        "expected_home_margin",
        "home_win_probability"
    ]
].loc[
    games_2026["week"] == 1
].copy()

print(
    week_1_predictions
    .round(3)
    .to_string(index=False)
)

print()
print(
    "Rest effect per day:",
    round(REST_POINT_VALUE, 3)
)

away_team home_team  neutral_strength_diff  home_field_adjustment  rest_adjustment  expected_home_margin  home_win_probability
       NE       SEA                  2.453                  1.764              0.0                 4.217                 0.617
       SF        LA                  2.396                  0.000              0.0                 2.396                 0.567
      CHI       CAR                 -4.481                  1.764              0.0                -2.718                 0.424
       TB       CIN                 -1.340                  1.764              0.0                 0.423                 0.512
       NO       DET                  5.575                  1.764              0.0                 7.338                 0.697
      BUF       HOU                 -2.006                  1.764              0.0                -0.243                 0.493
      BAL       IND                 -2.542                  1.764              0.0                -0.778       

## Predicted Winners

The team with the higher win probability is selected as the model's predicted winner.

These probabilities are not treated as certain outcomes. They will be used directly in the Monte Carlo simulation so that underdogs retain a realistic chance of winning each simulated game.

In [20]:
games_2026["predicted_winner"] = np.where(
    games_2026["home_win_probability"] >= 0.50,
    games_2026["home_team"],
    games_2026["away_team"]
)

games_2026["predicted_win_probability"] = np.maximum(
    games_2026["home_win_probability"],
    games_2026["away_win_probability"]
)

## Final Game Predictions

The final dataset contains one row for each 2026 regular season game, including the team strength matchup, schedule adjustments, expected margin, and win probabilities.

Positive expected margins favor the home team and negative margins favor the away team.

In [21]:
game_predictions_2026 = games_2026[
    [
        "game_id",
        "week",
        "gameday",
        "away_team",
        "home_team",
        "away_team_strength",
        "home_team_strength",
        "neutral_strength_diff",
        "home_field_adjustment",
        "rest_diff",
        "rest_adjustment",
        "expected_home_margin",
        "away_win_probability",
        "home_win_probability",
        "predicted_winner",
        "predicted_win_probability"
    ]
].copy()

In [22]:
game_predictions_2026.to_parquet(
    PROCESSED_DIR / "2026_game_predictions.parquet",
    index=False
)

In [23]:
print(
    f"Game predictions: {len(game_predictions_2026)} games"
)

print(
    f"Teams represented: "
    f"{len(set(game_predictions_2026['home_team']) | set(game_predictions_2026['away_team']))}"
)

print(
    f"Average home win probability: "
    f"{game_predictions_2026['home_win_probability'].mean():.3f}"
)

game_predictions_2026.head(10)

Game predictions: 272 games
Teams represented: 32
Average home win probability: 0.547


,game_id,week,gameday,away_team,home_team,away_team_strength,home_team_strength,neutral_strength_diff,home_field_adjustment,rest_diff,rest_adjustment,expected_home_margin,away_win_probability,home_win_probability,predicted_winner,predicted_win_probability
0,2026_01_NE_SEA,1,2026-09-09,NE,SEA,1.910188,4.363421,2.453233,1.763528,0,0.0,4.216761,0.383257,0.616743,SEA,0.616743
1,2026_01_SF_LA,1,2026-09-10,SF,LA,2.366788,4.762452,2.395664,0.000000,0,0.0,2.395664,0.433017,0.566983,LA,0.566983
2,2026_01_CHI_CAR,1,2026-09-13,CHI,CAR,-0.051934,-4.533242,-4.481308,1.763528,0,0.0,-2.717780,0.575887,0.424113,CHI,0.575887
3,2026_01_TB_CIN,1,2026-09-13,TB,CIN,0.601026,-0.739434,-1.340461,1.763528,0,0.0,0.423067,0.488117,0.511883,CIN,0.511883
4,2026_01_NO_DET,1,2026-09-13,NO,DET,-1.556053,4.018565,5.574618,1.763528,0,0.0,7.338146,0.302668,0.697332,DET,0.697332
5,2026_01_BUF_HOU,1,2026-09-13,BUF,HOU,4.816514,2.810133,-2.006381,1.763528,0,0.0,-0.242853,0.506822,0.493178,BUF,0.506822
6,2026_01_BAL_IND,1,2026-09-13,BAL,IND,3.074111,0.532226,-2.541885,1.763528,0,0.0,-0.778357,0.521855,0.478145,BAL,0.521855
7,2026_01_CLE_JAX,1,2026-09-13,CLE,JAX,-4.150395,1.713136,5.863531,1.763528,0,0.0,7.627059,0.295604,0.704396,JAX,0.704396
8,2026_01_ATL_PIT,1,2026-09-13,ATL,PIT,-1.331272,0.543902,1.875175,1.763528,0,0.0,3.638702,0.398886,0.601114,PIT,0.601114
9,2026_01_NYJ_TEN,1,2026-09-13,NYJ,TEN,-5.933726,-6.747344,-0.813618,1.763528,0,0.0,0.949910,0.473334,0.526666,TEN,0.526666
